# WybeCoder: Verified Imperative Code Generation — Concepts Notebook

**Paper:** *WybeCoder: Verified Imperative Code Generation* — Gloeckle et al., 2026  
**arXiv:** https://arxiv.org/abs/2603.29088  
**Project page:** https://facebookresearch.github.io/wybecoder  
**Official repo:** https://github.com/facebookresearch/wybecoder

---

This notebook does **not** require Lean 4 or an LLM API key.  
Instead, it walks through the core ideas in pure Python:

1. What a verification condition (VC) looks like, and what "discharging" it means
2. The hybrid SMT → Lean fallback loop (simulated with a toy prover)
3. The subgoal decomposition strategy
4. Invariant generation for a simple loop — the central challenge in the paper
5. Inference scaling: why parallel sub-agents outperform a single deep agent

## 1. Verification Conditions — What Are We Solving?

A **verification condition (VC)** is a logical formula that, if true, guarantees the annotated program is correct with respect to its specification.

For a `while` loop with invariant $I$, three VCs are generated automatically by the Loom compiler:

| VC | Meaning |
|----|---------|
| $\text{Pre} \Rightarrow I$ | Invariant holds before loop entry |
| $I \wedge \text{guard} \Rightarrow I[\text{after body}]$ | Invariant is preserved by each iteration |
| $I \wedge \neg\text{guard} \Rightarrow \text{Post}$ | Invariant + termination implies postcondition |

Below we represent VCs as simple Python callables that return `True/False`.

In [ ]:
from dataclasses import dataclass, field
from typing import Callable, List, Optional
import random

# A verification condition is just a named boolean function
@dataclass
class VC:
    name: str
    check: Callable[[], bool]   # True = valid (discharged), False = counter-example found

    def __repr__(self):
        return f"VC({self.name!r})"


# -----------------------------------------------------------------------
# Example: sum_to_n
#   method sumToN(n: Nat) return (s: Nat)
#     ensures s = n * (n + 1) / 2
#   Invariant candidate: s = i * (i + 1) / 2
# -----------------------------------------------------------------------

def make_sum_to_n_vcs(n_test: int = 20) -> List[VC]:
    """Generate the three loop VCs for sumToN with the correct invariant."""

    def vc_init():
        # Pre => I:  before loop, i=0, s=0 => 0 = 0*(0+1)/2 ✓
        i, s = 0, 0
        return s == i * (i + 1) // 2

    def vc_preservation():
        # I ∧ guard => I[i+1, s+i+1]
        for i in range(n_test):
            s = i * (i + 1) // 2   # valid state satisfying I
            s_new = s + (i + 1)
            i_new = i + 1
            if not (s_new == i_new * (i_new + 1) // 2):
                return False
        return True

    def vc_postcondition():
        # I ∧ ¬guard (i=n) => s = n*(n+1)/2
        for n in range(n_test):
            s = n * (n + 1) // 2
            if not (s == n * (n + 1) // 2):
                return False
        return True

    return [VC("init",         vc_init),
            VC("preservation", vc_preservation),
            VC("postcondition", vc_postcondition)]


vcs = make_sum_to_n_vcs()
for vc in vcs:
    result = vc.check()
    print(f"  {vc.name:20s}  {'✅ discharged' if result else '❌ failed'}")

## 2. The Hybrid SMT → Lean Fallback Loop

WybeCoder uses **cvc5** for automatic VC discharge. Goals cvc5 cannot solve in a time budget are escalated to an interactive Lean proof session.

We simulate this with two toy "solvers":
- `smt_solver` — succeeds on "easy" VCs (fast, automatic)
- `lean_solver` — succeeds on "hard" VCs with interactive hints (slower, higher success rate)

In [ ]:
import time

@dataclass
class SolverResult:
    vc_name: str
    solver: str       # "smt" | "lean" | "failed"
    success: bool
    latency_ms: float


def smt_solver(vc: VC, difficulty: float = 0.5) -> SolverResult:
    """
    Toy SMT solver. Succeeds with probability (1 - difficulty) immediately.
    Models cvc5 which handles linear arithmetic well but struggles with
    complex nonlinear invariants.
    """
    t0 = time.perf_counter()
    time.sleep(0.001)
    success = vc.check() and (random.random() > difficulty)
    return SolverResult(vc.name, "smt", success, (time.perf_counter() - t0) * 1000)


def lean_solver(vc: VC, difficulty: float = 0.3) -> SolverResult:
    """
    Toy Lean solver. Higher success rate than SMT but slower (interactive).
    Models a Lean REPL session with tactic hints from the LLM.
    """
    t0 = time.perf_counter()
    time.sleep(0.005)
    success = vc.check() and (random.random() > difficulty)
    return SolverResult(vc.name, "lean", success, (time.perf_counter() - t0) * 1000)


def hybrid_verify(vcs: List[VC], smt_difficulty=0.4, lean_difficulty=0.2) -> List[SolverResult]:
    """
    For each VC:
      1. Try SMT first (cheap, fast).
      2. On failure, escalate to Lean (expensive, more powerful).
    """
    results = []
    for vc in vcs:
        smt_result = smt_solver(vc, smt_difficulty)
        if smt_result.success:
            results.append(smt_result)
        else:
            lean_result = lean_solver(vc, lean_difficulty)
            results.append(lean_result)
    return results


random.seed(0)
results = hybrid_verify(vcs)
print(f"{'VC':<22} {'Solver':<6} {'Result':<15} {'Latency':>10}")
print("-" * 58)
for r in results:
    status = "✅ proved" if r.success else "❌ failed"
    print(f"{r.vc_name:<22} {r.solver:<6} {status:<15} {r.latency_ms:>8.2f} ms")

overall = all(r.success for r in results)
print(f"\nOverall: {'✅ VERIFIED' if overall else '❌ UNVERIFIED'}")

## 3. Subgoal Decomposition Strategy

The key WybeCoder insight: instead of one agent tackling all VCs sequentially, **split VCs into independent subgoals and dispatch parallel sub-agents**.

```
              Full problem (N VCs)
                     │
          ┌──────────┴──────────┐
       subgoal₁            subgoal₂  ...  subgoalₙ
          │                    │
      agent₁              agent₂    ...  agentₙ
    (k turns)           (k turns)
          │                    │
          └──────────┬──────────┘
               merge proofs
                     │
           reconstruct full proof
```

When sub-agents conflict (e.g. different invariants needed), **conflict-driven method modification** re-generates the implementation to reconcile them.

In [ ]:
import concurrent.futures

@dataclass
class SubgoalResult:
    subgoal_id: int
    success: bool
    turns_used: int


def run_subagent(subgoal_id: int, vc: VC, max_turns: int = 5,
                 success_prob_per_turn: float = 0.55) -> SubgoalResult:
    """
    Simulate a sub-agent trying to prove a single VC.
    Each turn it independently attempts the proof; stops on first success.
    """
    for turn in range(1, max_turns + 1):
        if vc.check() and random.random() < success_prob_per_turn:
            return SubgoalResult(subgoal_id, success=True, turns_used=turn)
    return SubgoalResult(subgoal_id, success=False, turns_used=max_turns)


def sequential_agent(vcs: List[VC], max_turns: int = 5) -> bool:
    """One agent processes all VCs in order; fails fast on first failure."""
    for vc in vcs:
        result = run_subagent(0, vc, max_turns)
        if not result.success:
            return False
    return True


def subgoal_decomposition(vcs: List[VC], max_turns: int = 5) -> bool:
    """Dispatch one sub-agent per VC in parallel; merge results."""
    with concurrent.futures.ThreadPoolExecutor() as executor:
        futures = {executor.submit(run_subagent, i, vc, max_turns): i
                   for i, vc in enumerate(vcs)}
        results = [f.result() for f in concurrent.futures.as_completed(futures)]
    return all(r.success for r in results)


# --- Monte Carlo comparison ---
N_TRIALS = 500
random.seed(42)

seq_successes   = sum(sequential_agent(vcs)       for _ in range(N_TRIALS))
decomp_successes = sum(subgoal_decomposition(vcs) for _ in range(N_TRIALS))

print(f"Sequential agent solve rate:      {seq_successes / N_TRIALS:.1%}  ({seq_successes}/{N_TRIALS})")
print(f"Subgoal decomposition solve rate: {decomp_successes / N_TRIALS:.1%}  ({decomp_successes}/{N_TRIALS})")
print(f"\nDecomposition advantage: +{(decomp_successes - seq_successes) / N_TRIALS:.1%}")

## 4. Invariant Generation — The Central Challenge

The hardest part of verified imperative code generation is **finding loop invariants**.

Given a `while` loop, an invariant $I$ must:
1. Hold before the loop
2. Be preserved by each iteration
3. Together with loop termination, imply the postcondition

We illustrate three invariant candidates for `sumToN` and check which ones are valid.

In [ ]:
def check_loop_invariant(invariant_fn, n_max=30):
    """
    Exhaustively check the three invariant conditions for the sumToN loop.

    Loop:
        i, s = 0, 0
        while i < n:
            s += i + 1
            i += 1
        # postcondition: s == n*(n+1)//2
    """
    issues = []

    for n in range(1, n_max + 1):
        # --- Condition 1: holds on entry (i=0, s=0) ---
        if not invariant_fn(i=0, s=0, n=n):
            issues.append(f"INIT FAIL: n={n}, i=0, s=0")
            break

        # --- Conditions 2 & 3: simulate the loop ---
        i, s = 0, 0
        while i < n:
            # Condition 2: preservation — I holds before body
            if not invariant_fn(i=i, s=s, n=n):
                issues.append(f"PRESERVATION FAIL: n={n}, i={i}, s={s}")
                break
            s += i + 1
            i += 1
        else:
            # Condition 3: postcondition — I ∧ ¬guard => post
            if not (invariant_fn(i=i, s=s, n=n) and s == n * (n + 1) // 2):
                issues.append(f"POSTCONDITION FAIL: n={n}, i={i}, s={s}")

    return issues


# Three candidate invariants an LLM might propose
candidates = {
    "Correct:   s = i*(i+1)//2":    lambda i, s, n: s == i * (i + 1) // 2,
    "Wrong:     s = i*i":           lambda i, s, n: s == i * i,
    "Weak:      s >= 0":            lambda i, s, n: s >= 0,
    "Incomplete: i <= n":           lambda i, s, n: i <= n,
}

print(f"{'Candidate':<38}  {'VCs pass?'}")
print("-" * 60)
for name, inv in candidates.items():
    issues = check_loop_invariant(inv)
    status = "✅ all 3 VCs discharged" if not issues else f"❌ {issues[0]}"
    print(f"{name:<38}  {status}")

## 5. Inference Scaling — Why Parallel Sub-agents Win

The paper's key empirical finding: **solve rate continues to improve with compute budget without plateauing**, unlike prior sequential approaches.

We model this as:
- **Sequential (pass@k)**: run $k$ independent single-agent attempts, take best.
- **Subgoal decomposition**: allocate $k$ sub-agents across subgoals in parallel.

The parallel approach wins because each subgoal is strictly easier than the full problem, so per-agent success probability is higher.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def pass_at_k(p_single: float, k: int) -> float:
    """Probability that at least one of k independent attempts succeeds."""
    return 1 - (1 - p_single) ** k


def decomp_solve_rate(p_subgoal: float, n_subgoals: int, k_per_subgoal: int) -> float:
    """
    Probability all subgoals are solved when each gets k_per_subgoal parallel agents.
    Assumes subgoals are independent (optimistic but illustrative).
    """
    p_one_subgoal = pass_at_k(p_subgoal, k_per_subgoal)
    return p_one_subgoal ** n_subgoals


# Model parameters (calibrated loosely to paper's results)
P_SINGLE_FULL     = 0.10   # single-agent success prob on full problem
P_SINGLE_SUBGOAL  = 0.40   # single-agent success prob on one subgoal (easier)
N_SUBGOALS        = 4      # VCs to prove (simplified)
BUDGET_RANGE      = range(1, 65)  # total number of model calls

seq_rates   = [pass_at_k(P_SINGLE_FULL, k) for k in BUDGET_RANGE]
decomp_rates = []
for total_k in BUDGET_RANGE:
    k_per = max(1, total_k // N_SUBGOALS)
    decomp_rates.append(decomp_solve_rate(P_SINGLE_SUBGOAL, N_SUBGOALS, k_per))

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(BUDGET_RANGE), seq_rates,   label=f"Sequential pass@k  (p={P_SINGLE_FULL})", color="steelblue")
ax.plot(list(BUDGET_RANGE), decomp_rates, label=f"Subgoal decomp.     ({N_SUBGOALS} subgoals, p={P_SINGLE_SUBGOAL})", color="tomato")
ax.axhline(0.74, color="gray", linestyle=":", alpha=0.7, label="Paper best: 74.1% (Verina)")
ax.set_xlabel("Total model calls (compute budget)")
ax.set_ylabel("Solve rate")
ax.set_title("Simulated inference scaling: sequential vs. subgoal decomposition")
ax.legend()
ax.set_ylim(0, 1.05)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nAt budget=32:")
print(f"  Sequential:         {pass_at_k(P_SINGLE_FULL, 32):.1%}")
print(f"  Subgoal decomp.:    {decomp_solve_rate(P_SINGLE_SUBGOAL, N_SUBGOALS, 32 // N_SUBGOALS):.1%}")

## 6. End-to-End Worked Example: Binary Search Verification

We walk through verifying a binary search implementation step by step — the same workflow WybeCoder automates with an LLM:

1. Write the implementation with annotated loop invariants
2. Generate verification conditions
3. Discharge VCs (manually here, automatically in WybeCoder)

In [ ]:
def binary_search_verified(arr: list, target: int) -> int:
    """
    Binary search with explicit invariant maintenance.

    Specification (Velvet-style pseudocode):
        requires: arr is sorted in non-decreasing order
        ensures:  result >= 0 -> arr[result] == target
        ensures:  result == -1 -> target not in arr

    Loop invariants:
        I1: 0 <= lo <= hi <= len(arr)
        I2: all arr[j] < target  for j in [0, lo)       -- left exclusion
        I3: all arr[j] > target  for j in [hi, len(arr)) -- right exclusion
    """
    lo, hi = 0, len(arr)

    # Check I1, I2, I3 hold on entry
    assert 0 <= lo <= hi <= len(arr), "I1 violated on entry"
    assert all(arr[j] < target for j in range(0, lo)), "I2 violated on entry"
    assert all(arr[j] > target for j in range(hi, len(arr))), "I3 violated on entry"

    while lo < hi:
        # Invariants hold at loop head — assert them
        assert 0 <= lo <= hi <= len(arr), f"I1 violated: lo={lo}, hi={hi}"
        assert all(arr[j] < target for j in range(0, lo)), f"I2 violated at lo={lo}"
        assert all(arr[j] > target for j in range(hi, len(arr))), f"I3 violated at hi={hi}"

        mid = (lo + hi) // 2
        if arr[mid] == target:
            return mid
        elif arr[mid] < target:
            lo = mid + 1   # maintains I2: arr[mid] < target, so extend left exclusion
        else:
            hi = mid       # maintains I3: arr[mid] > target, so shrink right bound

    # Loop exited: lo == hi, so by I2 and I3, target cannot be in arr
    return -1


# Correctness tests
import random as rng
rng.seed(1)

errors = 0
for _ in range(1000):
    arr = sorted(rng.sample(range(200), 20))
    target = rng.randint(0, 199)
    result = binary_search_verified(arr, target)

    if target in arr:
        assert result >= 0 and arr[result] == target, f"False negative: {target} in {arr}, got {result}"
    else:
        assert result == -1, f"False positive: {target} not in {arr}, got {result}"

print("✅ All 1,000 random tests passed — invariants maintained throughout.")

## Summary

| Concept | WybeCoder | This notebook |
|---------|-----------|---------------|
| Target language | Velvet (Lean 4 DSL) | Python pseudocode |
| VC generation | Loom compiler (automatic) | Handwritten |
| SMT solving | cvc5 | `smt_solver()` simulation |
| Interactive proofs | Lean 4 REPL | `lean_solver()` simulation |
| Invariant synthesis | LLM (Claude/GPT/Gemini) | Manual candidates |
| Subgoal parallelism | 16–128 sub-agents | `ThreadPoolExecutor` |
| Benchmarks | Verina (189), Clever-Loom (161) | Toy examples |

**Key takeaway:** WybeCoder's strength is that it turns software verification — traditionally a manual expert task — into a compute-scalable inference problem, with no plateau in solve rate as model calls increase.

---
*Notebook complete. See [`../README.md`](../README.md) for full paper notes.*